In [2]:
import pandas as pd

In [5]:
junction = pd.read_csv("movie_lookup.csv")
cast_member = pd.read_csv("cast_member_extract.csv")

cast_member_data = junction[['movieid', 'cast']].dropna().copy()

cast_member_data['cast'] = cast_member_data['cast'].str.split(',')
print(cast_member_data)

       movieid                                               cast
0            1                       [James Levine, Steve Martin]
1            2  [Eiko Masuyama, GorÃ´ Naya, IchirÃ´ Nagai, Kiy...
2            3           [Diane Lane, Frankie Muniz, Kevin Bacon]
3            4                                         [Ice Cube]
4            5      [James Spader, Peter Facinelli, Robin Tunney]
...        ...                                                ...
11359    11360  [Annabel O'Hagan, ChantÃ© Adams, Cleveland Ber...
11360    11361  [Adam Baldwin, Anna Paquin, Ashley Safer Fossi...
11361    11362  [Alex Hassell, Brendan Gleeson, Brian Thompson...
11362    11363  [Agnes Brekke, Constanza Gutierrez, Daniel Gim...
11363    11364  [Aki Kato, Clifton Collins Jr., Danny Garcia, ...

[7662 rows x 2 columns]


In [6]:
# fixing wrong mojibake pattern
cast_member_data['cast'] = cast_member_data['cast'].apply(
    lambda names: [name.encode('latin1', errors='ignore').decode('utf-8', errors='ignore') for name in names]
)

In [7]:
# check whether it worked
print(cast_member_data)

       movieid                                               cast
0            1                       [James Levine, Steve Martin]
1            2  [Eiko Masuyama, Gorô Naya, Ichirô Nagai, Kiyos...
2            3           [Diane Lane, Frankie Muniz, Kevin Bacon]
3            4                                         [Ice Cube]
4            5      [James Spader, Peter Facinelli, Robin Tunney]
...        ...                                                ...
11359    11360  [Annabel O'Hagan, Chanté Adams, Cleveland Bert...
11360    11361  [Adam Baldwin, Anna Paquin, Ashley Safer Fossi...
11361    11362  [Alex Hassell, Brendan Gleeson, Brian Thompson...
11362    11363  [Agnes Brekke, Constanza Gutierrez, Daniel Gim...
11363    11364  [Aki Kato, Clifton Collins Jr., Danny Garcia, ...

[7662 rows x 2 columns]


In [8]:
cast_member_data = cast_member_data.explode('cast')
print(cast_member_data)

       movieid             cast
0            1     James Levine
0            1     Steve Martin
1            2    Eiko Masuyama
1            2        Gorô Naya
1            2     Ichirô Nagai
...        ...              ...
11363    11364  Martin Bourdieu
11363    11364     Moisés Arias
11363    11364     Molly Parker
11363    11364      Ryan Barber
11363    11364  Vincent Francia

[48314 rows x 2 columns]


In [11]:
movie_cast = cast_member_data.merge(
    cast_member,
    left_on= 'cast',
    right_on= 'castname',
    how='left'
)

In [13]:
movie_cast.info

<bound method DataFrame.info of        movieid             cast   castid         castname
0            1     James Levine   8717.0     James Levine
1            1     Steve Martin   5278.0     Steve Martin
2            2    Eiko Masuyama  13610.0    Eiko Masuyama
3            2        Gorô Naya  13611.0        Gorô Naya
4            2     Ichirô Nagai  13612.0     Ichirô Nagai
...        ...              ...      ...              ...
48309    11364  Martin Bourdieu  12113.0  Martin Bourdieu
48310    11364     Moisés Arias   9236.0     Moisés Arias
48311    11364     Molly Parker    147.0     Molly Parker
48312    11364      Ryan Barber  12114.0      Ryan Barber
48313    11364  Vincent Francia  12115.0  Vincent Francia

[48314 rows x 4 columns]>

In [12]:
# check for nulls
print(movie_cast['castid'].isna().sum())

18


In [14]:
# check for null values
movie_cast[movie_cast['castid'].isna()]

,movieid,cast,castid,castname
25225,9395,,NaN,NaN
26028,9460,,NaN,NaN
26280,9480,,NaN,NaN
27514,9581,,NaN,NaN
29972,9777,,NaN,NaN
30727,9840,,NaN,NaN
30772,9843,,NaN,NaN
32167,9950,,NaN,NaN
32742,9996,,NaN,NaN
33269,10040,,NaN,NaN


In [15]:
# check whether they're empty strings (no null value so would've survived the earlier dropna)
movie_cast[movie_cast['castid'].isna()]['cast'].apply(repr)

25225    ''
26028    ''
26280    ''
27514    ''
29972    ''
30727    ''
30772    ''
32167    ''
32742    ''
33269    ''
34288    ''
34785    ''
36461    ''
39602    ''
39747    ''
40965    ''
45265    ''
45419    ''
Name: cast, dtype: str

In [16]:
# filtering out the 18 empty strings
movie_cast = movie_cast[movie_cast['cast'].str.strip() != '']

In [17]:
# double check
print(movie_cast['castid'].isna().sum())

0


In [18]:
movie_cast.info()
movie_cast.isnull().sum().sum()

movie_cast = movie_cast[['movieid','castid']]

movie_cast = movie_cast.drop_duplicates(
    subset=['movieid', 'castid']
)

movie_cast['castid'] = movie_cast['castid'].astype(int) # change type to int

movie_cast.to_csv("movie_cast.csv",index=False)

<class 'pandas.DataFrame'>
Index: 48296 entries, 0 to 48313
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   movieid   48296 non-null  int64  
 1   cast      48296 non-null  str    
 2   castid    48296 non-null  float64
 3   castname  48296 non-null  str    
dtypes: float64(1), int64(1), str(2)
memory usage: 3.1 MB


In [19]:
movie_cast.columns

Index(['movieid', 'castid'], dtype='str')

In [20]:
movie_cast.head()
movie_cast.shape

(48284, 2)